In [10]:
!pip install torch_geometric

# Imports & Device Setup

In [ ]:
import os
import itertools
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

from torch_geometric.nn import GCNConv
from torch_geometric.utils import add_self_loops

from tqdm import tqdm

import torch._dynamo
torch._dynamo.config.suppress_errors = True

# Output directory
OUT_DIR = "ROSMAP_output"
os.makedirs(OUT_DIR, exist_ok=True)

#  Device & CUDA tuning
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IS_CUDA = device.type == 'cuda'

print(f'Using device: {device}')

if IS_CUDA:
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    torch.backends.cudnn.benchmark        = True
    torch.backends.cudnn.deterministic    = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

print('Cell 1 complete.\n')

Using device: cuda
  GPU  : Tesla T4
  VRAM : 15.64 GB
Cell 1 complete.



# Data Loading & Preprocessing

In [ ]:

# Load raw CSVs
geneexp_train = pd.read_csv('1_tr.csv')
geneexp_test  = pd.read_csv('1_te.csv')

myth_train    = pd.read_csv('2_tr.csv')
myth_test     = pd.read_csv('2_te.csv')

mirna_train   = pd.read_csv('3_tr.csv')
mirna_test    = pd.read_csv('3_te.csv')

label_train   = pd.read_csv('labels_tr.csv')
label_test    = pd.read_csv('labels_te.csv')

# Fit scalers on TRAIN only — transform both splits
# Prevents data leakage from test into train normalisation
scaler_expr  = StandardScaler()
scaler_mirna = StandardScaler()
scaler_meth  = StandardScaler()

X_expr_tr  = torch.tensor(
    scaler_expr.fit_transform(geneexp_train.values), dtype=torch.float)
X_mirna_tr = torch.tensor(
    scaler_mirna.fit_transform(mirna_train.values),  dtype=torch.float)
X_meth_tr  = torch.tensor(
    scaler_meth.fit_transform(myth_train.values),    dtype=torch.float)
y_tr       = torch.tensor(
    label_train.values.squeeze(), dtype=torch.float).reshape(-1, 1)

X_expr_te  = torch.tensor(
    scaler_expr.transform(geneexp_test.values),  dtype=torch.float)
X_mirna_te = torch.tensor(
    scaler_mirna.transform(mirna_test.values),   dtype=torch.float)
X_meth_te  = torch.tensor(
    scaler_meth.transform(myth_test.values),     dtype=torch.float)
y_te       = torch.tensor(
    label_test.values.squeeze(), dtype=torch.float).reshape(-1, 1)

#  Concatenate into full patient graph
X_expr  = torch.cat([X_expr_tr,  X_expr_te],  dim=0)
X_mirna = torch.cat([X_mirna_tr, X_mirna_te], dim=0)
X_meth  = torch.cat([X_meth_tr,  X_meth_te],  dim=0)
y       = torch.cat([y_tr, y_te], dim=0)

n_train = len(label_train)
n_test  = len(label_test)
n_total = n_train + n_test
n_feats = X_expr.shape[1]

# Sanity checks
assert X_expr.shape  == (n_total, n_feats), f"X_expr  shape mismatch: {X_expr.shape}"
assert X_mirna.shape == (n_total, n_feats), f"X_mirna shape mismatch: {X_mirna.shape}"
assert X_meth.shape  == (n_total, n_feats), f"X_meth  shape mismatch: {X_meth.shape}"
assert y.shape       == (n_total, 1),       f"y shape mismatch:       {y.shape}"

print('Data loaded & preprocessed.')
print(f'  n_train : {n_train}')
print(f'  n_test  : {n_test}')
print(f'  n_total : {n_total}')
print(f'  n_feats : {n_feats} per modality × 3 modalities')
print(f'  Labels  : {int(y.sum().item())} positive  /  '
      f'{int((1 - y).sum().item())} negative')
print('Cell 2 complete.\n')

Data loaded & preprocessed.
  n_train : 244
  n_test  : 105
  n_total : 349
  n_feats : 200 per modality × 3 modalities
  Labels  : 181 positive  /  168 negative
Cell 2 complete.



# Patient Graph Construction

In [ ]:
K_NEIGHBOURS = 10

print("Constructing patient similarity graph...")

# Concatenate all modalities for similarity computation
features_np = torch.cat([X_expr, X_mirna, X_meth], dim=1).numpy()  # (244, 3F)

# Check for NaN/Inf in features
if np.any(np.isnan(features_np)) or np.any(np.isinf(features_np)):
    print("  WARNING: NaN/Inf found in features. Cleaning...")
    features_np = np.nan_to_num(features_np, nan=0.0, posinf=1.0, neginf=-1.0)

print(f"  Features shape: {features_np.shape}")

# Compute similarity matrix
sim_matrix = cosine_similarity(features_np)

# Remove self-similarity (set diagonal to 0)
np.fill_diagonal(sim_matrix, 0)

print(f"  Similarity matrix shape: {sim_matrix.shape}")
print(f"  Similarity range: [{sim_matrix.min():.4f}, {sim_matrix.max():.4f}]")

edges, weights = [], []

for i in range(sim_matrix.shape[0]):
    # Get top-k neighbours (excluding self)

    n_neighbours = min(K_NEIGHBOURS, sim_matrix.shape[1] - 1)

    # Get indices of top-k neighbours

    top_k_idx = np.argsort(sim_matrix[i])[-n_neighbours:]

    for j in top_k_idx:
        if j != i:
            edges.append([i, int(j)])
            weights.append(float(sim_matrix[i, j]))

print(f"  Total edges (undirected): {len(edges)}")

if len(edges) == 0:
    print("  ERROR: No edges created! Using random graph fallback...")
    # Fallback to random edges if similarity graph is empty
    import random
    for i in range(n_total):
        for _ in range(K_NEIGHBOURS):
            j = random.randint(0, n_total - 1)
            if j != i:
                edges.append([i, j])
                weights.append(0.5)

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(weights, dtype=torch.float)

print(f"  Edge index shape: {edge_index.shape}")
print(f"  Edge weight range: [{edge_weight.min():.4f}, {edge_weight.max():.4f}]")

# Validate edge indices
assert edge_index.min() >= 0, f"Negative edge index: {edge_index.min()}"
assert edge_index.max() < n_total, f"Edge index {edge_index.max()} >= n_total {n_total}"

# Add self-loops (fill value = 1.0)
edge_index, edge_weight = add_self_loops(
    edge_index, edge_weight, fill_value=1.0,
    num_nodes=n_total
)

print(f"\nFinal graph statistics:")
print(f"  k-neighbours : {K_NEIGHBOURS}")
print(f"  Nodes        : {n_total}")
print(f"  Edges        : {edge_index.shape[1]}  (including self-loops)")
print(f"  Edge index   : [{edge_index.min()}, {edge_index.max()}]")
print('Cell 3 complete.\n')

Constructing patient similarity graph...
  Features shape: (349, 600)
  Similarity matrix shape: (349, 349)
  Similarity range: [-0.6376, 0.7060]
  Total edges (undirected): 3490
  Edge index shape: torch.Size([2, 3490])
  Edge weight range: [0.1417, 0.7060]

Final graph statistics:
  k-neighbours : 10
  Nodes        : 349
  Edges        : 3839  (including self-loops)
  Edge index   : [0, 348]
Cell 3 complete.



# Move Tensors to GPU with Error Handling

In [ ]:
# Debug check before moving to GPU
print("Pre-GPU transfer checks:")
print(f"  X_expr dtype: {X_expr.dtype}, shape: {X_expr.shape}")
print(f"  X_mirna dtype: {X_mirna.dtype}, shape: {X_mirna.shape}")
print(f"  X_meth dtype: {X_meth.dtype}, shape: {X_meth.shape}")
print(f"  edge_index dtype: {edge_index.dtype}, shape: {edge_index.shape}")
print(f"  edge_weight dtype: {edge_weight.dtype}, shape: {edge_weight.shape}")
print(f"  y dtype: {y.dtype}, shape: {y.shape}")

# Validate edge_index bounds
print("\nValidating edge_index bounds...")
print(f"  edge_index max: {edge_index.max().item()}")
print(f"  edge_index min: {edge_index.min().item()}")
print(f"  Expected nodes: {n_total}")

if edge_index.max().item() >= n_total:
    print(f"  WARNING: edge_index has indices >= {n_total}!")
    print(f"  Clamping edge_index to [{0}, {n_total-1}]...")
    edge_index = edge_index.clamp(0, n_total - 1)

if edge_index.min().item() < 0:
    print(f"  WARNING: edge_index has negative indices!")
    print(f"  Clamping edge_index to [0, {n_total-1}]...")
    edge_index = edge_index.clamp(0, n_total - 1)

# Validate no NaN or Inf in features
print("\nChecking for NaN/Inf values:")
for name, tensor in [("X_expr", X_expr), ("X_mirna", X_mirna),
                     ("X_meth", X_meth), ("edge_weight", edge_weight),
                     ("y", y)]:
    nan_count = torch.isnan(tensor).sum().item()
    inf_count = torch.isinf(tensor).sum().item()
    print(f"  {name}: NaN={nan_count}, Inf={inf_count}")
    if nan_count > 0 or inf_count > 0:
        print(f"    Fixing {name}...")
        tensor = torch.nan_to_num(tensor, nan=0.0, posinf=1.0, neginf=-1.0)

# Ensure correct dtypes
X_expr  = X_expr.float()
X_mirna = X_mirna.float()
X_meth  = X_meth.float()
y       = y.float()
edge_index = edge_index.long()
edge_weight = edge_weight.float()


if IS_CUDA:
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    print("\nCUDA_LAUNCH_BLOCKING enabled for debugging")


if IS_CUDA:
    torch.cuda.empty_cache()
    print(f"GPU memory cleared. Current allocated: "
          f"{torch.cuda.memory_allocated()/1e9:.2f} GB")


print("\nMoving tensors to GPU...")

try:
    X_list_gpu = [
        X_expr.to(device, non_blocking=True),
        X_mirna.to(device, non_blocking=True),
        X_meth.to(device, non_blocking=True),
    ]
    print(" X_list_gpu moved successfully")
except RuntimeError as e:
    print(f" Error moving X tensors: {e}")
    print(" Attempting without non_blocking...")
    X_list_gpu = [
        X_expr.to(device, non_blocking=False),
        X_mirna.to(device, non_blocking=False),
        X_meth.to(device, non_blocking=False),
    ]

try:
    edge_idx_gpu = edge_index.to(device, non_blocking=True)
    edge_wt_gpu  = edge_weight.to(device, non_blocking=True)
    print(" Edge tensors moved successfully")
except RuntimeError as e:
    print(f" Error moving edge tensors: {e}")
    print("  Attempting without non_blocking...")
    edge_idx_gpu = edge_index.to(device, non_blocking=False)
    edge_wt_gpu  = edge_weight.to(device, non_blocking=False)

try:
    y_gpu = y.to(device, non_blocking=True).view(-1)
    print(" y moved successfully")
except RuntimeError as e:
    print(f" Error moving y: {e}")
    y_gpu = y.to(device, non_blocking=False).view(-1)

# Verify GPU transfer
print("\nVerifying GPU transfer:")
for i, x in enumerate(X_list_gpu):
    print(f"  X_list_gpu[{i}]: device={x.device}, shape={x.shape}, "
          f"dtype={x.dtype}")
print(f"  edge_idx_gpu: device={edge_idx_gpu.device}, shape={edge_idx_gpu.shape}")
print(f"  edge_wt_gpu: device={edge_wt_gpu.device}, shape={edge_wt_gpu.shape}")
print(f"  y_gpu: device={y_gpu.device}, shape={y_gpu.shape}")

# Create index tensors
train_idx = torch.arange(n_train, dtype=torch.long, device=device)
te_global = torch.arange(n_train, n_total, dtype=torch.long, device=device)

print(f"\nIndex tensors:")
print(f"  train_idx: device={train_idx.device}, shape={train_idx.shape}")
print(f"  te_global: device={te_global.device}, shape={te_global.shape}")

# Pre-fetch test labels
y_te_gpu = y_gpu[te_global]

# Input dimensions
in_dims = [X.shape[1] for X in X_list_gpu]

print(f"\nInput dimensions: {in_dims}")

print("\nPerforming GPU sanity check...")
try:
    with torch.no_grad():
        # Simple tensor operation to verify GPU is working
        test_tensor = torch.ones(10, device=device)
        test_result = test_tensor.sum().item()
    print(f"  GPU sanity check passed (1*10 = {test_result})")
except RuntimeError as e:
    print(f" GPU sanity check failed: {e}")
    print("  Check CUDA installation and GPU status")

print("\nAll tensors moved to device successfully.")
print(f"  train_idx : {len(train_idx)} nodes  (indices 0 to {len(train_idx)-1})")
print(f"  te_global : {len(te_global)} nodes  (indices {n_train} to {n_total-1})")
print(f"  in_dims   : {in_dims}")
print('Cell 4 complete.\n')

# Disable CUDA_LAUNCH_BLOCKING after debugging
if IS_CUDA:
    if 'CUDA_LAUNCH_BLOCKING' in os.environ:
        del os.environ['CUDA_LAUNCH_BLOCKING']
    print("CUDA_LAUNCH_BLOCKING disabled (set only for debugging)")

Pre-GPU transfer checks:
  X_expr dtype: torch.float32, shape: torch.Size([349, 200])
  X_mirna dtype: torch.float32, shape: torch.Size([349, 200])
  X_meth dtype: torch.float32, shape: torch.Size([349, 200])
  edge_index dtype: torch.int64, shape: torch.Size([2, 3839])
  edge_weight dtype: torch.float32, shape: torch.Size([3839])
  y dtype: torch.float32, shape: torch.Size([349, 1])

Validating edge_index bounds...
  edge_index max: 348
  edge_index min: 0
  Expected nodes: 349

Checking for NaN/Inf values:
  X_expr: NaN=0, Inf=0
  X_mirna: NaN=0, Inf=0
  X_meth: NaN=0, Inf=0
  edge_weight: NaN=0, Inf=0
  y: NaN=0, Inf=0

CUDA_LAUNCH_BLOCKING enabled for debugging
GPU memory cleared. Current allocated: 0.00 GB

Moving tensors to GPU...
 X_list_gpu moved successfully
 Edge tensors moved successfully
 y moved successfully

Verifying GPU transfer:
  X_list_gpu[0]: device=cuda:0, shape=torch.Size([349, 200]), dtype=torch.float32
  X_list_gpu[1]: device=cuda:0, shape=torch.Size([349, 200])

# Shared Components (EarlyStopping, Metrics, Loss)

In [ ]:
# GPU-resident Early Stopping
class EarlyStopping:
    """
    Checkpoints remain on GPU via {k: v.clone()} — no deepcopy, no CPU transfer.
    mode='max'  →  AUC / F1 / Accuracy
    mode='min'  →  loss
    """

    def __init__(self, patience: int = 40, min_delta: float = 1e-4,
                 mode: str = 'max'):
        assert mode in ('max', 'min')
        self.patience  = patience
        self.min_delta = min_delta
        self.mode      = mode
        self.reset()

    def reset(self):
        self.best_score    = None
        self.counter       = 0
        self.best_state    = None
        self.best_epoch    = 0
        self.stopped_epoch = 0

    def step(self, score: float, model: nn.Module, epoch: int) -> bool:
        if self._is_better(score):
            self.best_score = score
            self.counter    = 0
            self.best_epoch = epoch
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1

        if self.counter >= self.patience:
            self.stopped_epoch = epoch
            return True
        return False

    def restore_best(self, model: nn.Module):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

    def _is_better(self, score: float) -> bool:
        if self.best_score is None:
            return True
        if self.mode == 'max':
            return score > self.best_score + self.min_delta
        return score < self.best_score - self.min_delta


# Binary metrics (GPU tensors → one CPU sync for AUC)
@torch.inference_mode()
def gpu_metrics(probs: torch.Tensor, labels: torch.Tensor,
                threshold: float = 0.5):
    """
    Returns: auc, f1, accuracy, precision, recall
    All operations on GPU except roc_auc_score (sklearn).
    """
    # FIX: If raw logits are passed (values < 0 or > 1), convert them to probabilities first
    if probs.min() < 0 or probs.max() > 1:
        probs = torch.sigmoid(probs)

    probs_np  = probs.cpu().numpy()
    labels_np = labels.cpu().numpy()

    pred_bin = (probs >= threshold).long()
    labels_l = labels.long()

    auc  = roc_auc_score(labels_np, probs_np)

    tp   = ((pred_bin == 1) & (labels_l == 1)).sum().float()
    fp   = ((pred_bin == 1) & (labels_l == 0)).sum().float()
    fn   = ((pred_bin == 0) & (labels_l == 1)).sum().float()
    tn   = ((pred_bin == 0) & (labels_l == 0)).sum().float()

    acc  = ((tp + tn) / (tp + fp + fn + tn)).item()
    prec = (tp / (tp + fp + 1e-8)).item()
    rec  = (tp / (tp + fn + 1e-8)).item()
    f1   = 2 * prec * rec / (prec + rec + 1e-8)

    return auc, f1, acc, prec, rec


# Focal Loss
class FocalLoss(nn.Module):
    """
    Binary focal loss.
    γ controls focus on hard examples; α balances positive class.
    """

    def __init__(self, gamma: float = 2.0, alpha: float = 0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, preds: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        preds   = preds.float().view(-1)
        targets = targets.float().view(-1)

        # FIX: Use binary_cross_entropy_with_logits which safely handles raw unbound inputs
        bce     = F.binary_cross_entropy_with_logits(preds, targets, reduction='none')

        pt      = torch.exp(-bce)
        loss    = self.alpha * (1.0 - pt) ** self.gamma * bce
        return loss.mean()


print('Shared components defined:  EarlyStopping · gpu_metrics · FocalLoss')
print('Cell 5 complete.\n')

Shared components defined:  EarlyStopping · gpu_metrics · FocalLoss
Cell 5 complete.



# Model Construction

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
import numpy as np

# Fuzzy Covering Formation
def fuzzy_covering(adj, k=5):
    """
    Create fuzzy coverings from adjacency matrix.
    Returns list of k-nearest neighbors with their similarity weights.
    """
    coverings = []
    for i in range(adj.shape[0]):
        # Get top-k neighbors (excluding self)
        topk_idx = np.argsort(adj[i])[-k:]
        covering = [(int(topk_idx[j]), float(adj[i, topk_idx[j]])) for j in range(k)]
        coverings.append(covering)
    return coverings

def coverings_to_edge_index(coverings, return_weights=True):
    """
    Convert fuzzy coverings to PyTorch Geometric edge format.
    """
    edges = []
    weights = []

    for i, covering in enumerate(coverings):
        for neighbor, weight in covering:
            edges.append([i, neighbor])
            weights.append(weight)

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    if return_weights:
        edge_weight = torch.tensor(weights, dtype=torch.float32)
        return edge_index, edge_weight
    return edge_index

def build_fuzzy_graph(X, k=10):
    """
    Build fuzzy graph from feature matrix using cosine similarity.

    Args:
        X: Feature matrix [num_samples, num_features] (numpy array or torch tensor)
        k: Number of nearest neighbors for fuzzy covering

    Returns:
        edge_index: Edge indices [2, num_edges]
        edge_weight: Fuzzy membership weights [num_edges]
        coverings: List of fuzzy coverings
    """
    from sklearn.metrics.pairwise import cosine_similarity

    # Convert to numpy if torch tensor
    if isinstance(X, torch.Tensor):
        X = X.numpy()

    # Compute similarity matrix
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)

    # Create fuzzy coverings
    coverings = fuzzy_covering(sim, k=k)

    # Convert to edge format
    edge_index, edge_weight = coverings_to_edge_index(coverings)

    return edge_index, edge_weight, coverings

# Fuzzy Graph Convolution Layer
class FuzzyCoverConv(MessagePassing):
    """
    Fuzzy covering-based graph convolution with learnable aggregation.
    """
    def __init__(self, in_channels, out_channels, heads=1, dropout=0.0):
        super().__init__(aggr='add', node_dim=0)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.dropout = dropout

        # Linear transformation
        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)

        # Attention mechanism for fuzzy weighting
        self.att_src = nn.Parameter(torch.Tensor(1, heads, out_channels))
        self.att_dst = nn.Parameter(torch.Tensor(1, heads, out_channels))

        self.bias = nn.Parameter(torch.Tensor(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_weight=None):
        """
        Args:
            x: Node features [num_nodes, in_channels]
            edge_index: Edge indices [2, num_edges]
            edge_weight: Fuzzy membership weights [num_edges]
        """
        # Linear transformation
        x = self.lin(x).view(-1, self.heads, self.out_channels)

        # Propagate messages
        out = self.propagate(edge_index, x=x, edge_weight=edge_weight)

        # Average over heads
        out = out.mean(dim=1)
        out = out + self.bias

        return out

    def message(self, x_i, x_j, edge_weight, index, ptr, size_i):
        """
        Compute messages with fuzzy attention.
        """
        # Compute attention scores
        alpha = (x_i * self.att_src).sum(dim=-1) + (x_j * self.att_dst).sum(dim=-1)
        alpha = F.leaky_relu(alpha, 0.2)

        # Incorporate fuzzy edge weights
        if edge_weight is not None:
            alpha = alpha * edge_weight.view(-1, 1)

        # Softmax normalization
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        # Weighted message
        return x_j * alpha.unsqueeze(-1)

# Fuzzy Graph Convolutional Network
class FGC_GNN(nn.Module):
    """
    Fuzzy Graph Convolutional Network for multi-omics classification.
    Integrates fuzzy covering-based convolutions with attention mechanisms.

    This model is dataset-independent and can be used with any preprocessed data.
    """
    def __init__(self, in_dims, hidden_channels, out_channels,
                 heads=4, dropout=0.3, use_fuzzy_conv=True):
        """
        Args:
            in_dims: List of input dimensions for each omics layer
            hidden_channels: Hidden dimension size
            out_channels: Number of output classes
            heads: Number of attention heads
            dropout: Dropout probability
            use_fuzzy_conv: Use FuzzyCoverConv if True, otherwise GCNConv
        """
        super().__init__()
        self.dropout_p = dropout
        self.use_fuzzy_conv = use_fuzzy_conv
        self.num_omics = len(in_dims)

        # Per-omics graph convolution branches
        if use_fuzzy_conv:
            self.omics_gcns = nn.ModuleList([
                FuzzyCoverConv(d, hidden_channels, heads=heads, dropout=dropout)
                for d in in_dims
            ])
        else:
            from torch_geometric.nn import GCNConv
            self.omics_gcns = nn.ModuleList([
                GCNConv(d, hidden_channels, improved=True)
                for d in in_dims
            ])

        # Residual projections
        self.res_proj = nn.ModuleList([
            nn.Linear(d, hidden_channels) for d in in_dims
        ])

        # Fuzzy attention gates per omics branch
        self.attn = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels // 4),
                nn.ReLU(),
                nn.Linear(hidden_channels // 4, 1),
                nn.Sigmoid()
            ) for _ in range(self.num_omics)
        ])

        # Cross-omics attention
        self.cross_attn = nn.MultiheadAttention(
            hidden_channels, num_heads=heads, dropout=dropout, batch_first=True
        )

        # Fusion layers
        self.fc1 = nn.Linear(hidden_channels * self.num_omics, hidden_channels * 2)
        self.bn1 = nn.BatchNorm1d(hidden_channels * 2)

        self.fc2 = nn.Linear(hidden_channels * 2, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.fc3 = nn.Linear(hidden_channels, out_channels)

        # Layer normalization for each omics
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_channels) for _ in range(self.num_omics)
        ])

    def forward(self, X_list, edge_index, edge_weight=None):
        """
        Args:
            X_list: List of feature tensors, one per omics [num_nodes, features_i]
            edge_index: Edge indices from fuzzy covering [2, num_edges]
            edge_weight: Fuzzy membership weights [num_edges]

        Returns:
            out: Classification logits [num_nodes, out_channels]
        """
        omics_embeddings = []

        # Process each omics layer
        for i, (gcn, res, attn, ln, X) in enumerate(
            zip(self.omics_gcns, self.res_proj, self.attn, self.layer_norms, X_list)
        ):
            # Graph convolution with fuzzy weights
            h = gcn(X, edge_index, edge_weight=edge_weight)

            # Residual connection
            h_res = res(X)
            h = h + h_res

            # Layer normalization
            h = ln(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout_p, training=self.training)

            # Fuzzy attention gate
            attn_weight = attn(h)
            h = attn_weight * h

            omics_embeddings.append(h)

        # Stack for cross-omics attention [num_nodes, num_omics, hidden_channels]
        omics_stack = torch.stack(omics_embeddings, dim=1)

        # Apply cross-omics attention
        attn_out, _ = self.cross_attn(omics_stack, omics_stack, omics_stack)

        # Residual connection for attention
        omics_stack = omics_stack + attn_out

        # Flatten for fusion
        h = omics_stack.reshape(omics_stack.size(0), -1)

        # Multi-layer fusion with residual
        h1 = F.relu(self.bn1(self.fc1(h)))
        h1 = F.dropout(h1, p=self.dropout_p, training=self.training)

        h2 = F.relu(self.bn2(self.fc2(h1)))
        h2 = F.dropout(h2, p=self.dropout_p, training=self.training)

        # Output layer
        out = self.fc3(h2)

        return out


# Example Usage (For Testing Only)
def test_model_with_dummy_data():
    """
    Test the model with dummy data to verify it works correctly.
    This is just for demonstration - replace with your actual data.
    """
    print("Testing FGC_GNN with dummy data...")

    # Create dummy data
    num_samples = 100
    num_features = 500
    num_omics = 4

    # Split features into omics
    feature_splits = np.array_split(np.arange(num_features), num_omics)

    # Create random feature matrix
    X_full = np.random.randn(num_samples, num_features)

    # Split into omics-specific feature tensors
    X_list = [
        torch.tensor(X_full[:, idx], dtype=torch.float32)
        for idx in feature_splits
    ]

    print(f"\nDummy data created:")
    print(f"  Samples: {num_samples}")
    print(f"  Total features: {num_features}")
    print(f"  Omics layers: {num_omics}")
    for i, x in enumerate(X_list):
        print(f"    Omics {i+1}: {x.shape}")

    # Build fuzzy graph
    print("\nBuilding fuzzy graph...")
    edge_index, edge_weight, coverings = build_fuzzy_graph(X_full, k=10)

    print(f"  Fuzzy coverings: {len(coverings)}")
    print(f"  Edge index shape: {edge_index.shape}")
    print(f"  Edge weights shape: {edge_weight.shape}")

    # Initialize model
    in_dims = [x.shape[1] for x in X_list]
    model = FGC_GNN(
        in_dims=in_dims,
        hidden_channels=128,
        out_channels=2,  # Binary classification
        heads=4,
        dropout=0.3,
        use_fuzzy_conv=True
    )

    print("\n" + "="*80)
    print("Model Architecture:")
    print("="*80)
    print(model)
    print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Test forward pass
    print("\nTesting forward pass...")
    model.eval()
    with torch.no_grad():
        output = model(X_list, edge_index, edge_weight)
        print(f"  Output shape: {output.shape}")
        print(f"  Sample predictions:\n{output[:5]}")

    print("\n✓ Model test completed successfully!")

    return model, X_list, edge_index, edge_weight


if __name__ == "__main__":
    # Run test with dummy data
    model, X_list, edge_index, edge_weight = test_model_with_dummy_data()




Testing FGC_GNN with dummy data...

Dummy data created:
  Samples: 100
  Total features: 500
  Omics layers: 4
    Omics 1: torch.Size([100, 125])
    Omics 2: torch.Size([100, 125])
    Omics 3: torch.Size([100, 125])
    Omics 4: torch.Size([100, 125])

Building fuzzy graph...
  Fuzzy coverings: 100
  Edge index shape: torch.Size([2, 1000])
  Edge weights shape: torch.Size([1000])

Model Architecture:
FGC_GNN(
  (omics_gcns): ModuleList(
    (0-3): 4 x FuzzyCoverConv(125, 128)
  )
  (res_proj): ModuleList(
    (0-3): 4 x Linear(in_features=125, out_features=128, bias=True)
  )
  (attn): ModuleList(
    (0-3): 4 x Sequential(
      (0): Linear(in_features=128, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=1, bias=True)
      (3): Sigmoid()
    )
  )
  (cross_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
  )
  (fc1): Linear(in_features=512, out_features=256, bias=Tr

# Training Helper Used by Both CV Search and Final Training

In [ ]:

def build_and_train(
    hp          : dict,
    tr_idx      : torch.Tensor,
    val_idx     : torch.Tensor,
    es_patience : int  = 30,
    eval_every  : int  = 5,
    verbose     : bool = False,
) -> float:
    """
    Build a fresh FGC_GNN, train on `tr_idx` nodes,
    monitor early stopping on `val_idx` nodes.

    Both index tensors must already reside on `device`.

    Parameters
    ----------
    hp           : dict with keys
                   hidden_channels, lr, weight_decay, dropout,
                   focal_gamma, focal_alpha, epochs
    tr_idx       : 1-D LongTensor — training node indices
    val_idx      : 1-D LongTensor — validation node indices
    es_patience  : early-stopping patience (in eval checks)
    eval_every   : run eval every N epochs
    verbose      : print per-epoch progress

    Returns
    -------
    best_val_auc : float
    """

    epochs = hp['epochs']

    # Model
    model = FGC_GNN(
        in_dims         = in_dims,
        hidden_channels = hp['hidden_channels'],
        out_channels    = 1,
        dropout         = hp['dropout'],
    ).to(device)

    # Optimiser & scheduler
    optimizer  = optim.AdamW(
        model.parameters(),
        lr           = hp['lr'],
        weight_decay = hp['weight_decay'],
    )
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion  = FocalLoss(gamma=hp['focal_gamma'], alpha=hp['focal_alpha'])
    amp_scaler = torch.amp.GradScaler('cuda', enabled=IS_CUDA)
    es         = EarlyStopping(patience=es_patience, mode='max')

    y_tr_loc  = y_gpu[tr_idx]
    y_val_loc = y_gpu[val_idx]

    # Training loop
    for epoch in range(epochs):

        # Train
        model.train()
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=IS_CUDA):
            out = model(X_list_gpu, edge_idx_gpu, edge_wt_gpu)

        loss = criterion(out[tr_idx], y_tr_loc)
        amp_scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        amp_scaler.step(optimizer)
        amp_scaler.update()
        scheduler.step()

        # Eval (throttled)
        if (epoch + 1) % eval_every == 0 or epoch == epochs - 1:
            with torch.inference_mode():
                out_e     = model(X_list_gpu, edge_idx_gpu, edge_wt_gpu)
                val_probs = out_e[val_idx].view(-1)

            val_auc, val_f1, val_acc, *_ = gpu_metrics(val_probs, y_val_loc)

            if verbose:
                print(f'  ep {epoch+1:>4}  loss={loss.item():.4f}  '
                      f'val_AUC={val_auc:.4f}  val_F1={val_f1:.4f}')

            if es.step(val_auc, model, epoch):
                break

    return es.best_score if es.best_score is not None else 0.0


print('build_and_train() defined.')
print('Cell 7 complete.\n')

build_and_train() defined.
Cell 7 complete.



# 5-Fold CV Hyperparameter Search

In [9]:
PARAM_GRID = {
    'hidden_channels': [128, 256, 512],
    'lr'             : [1e-3, 2e-4],
    'weight_decay'   : [1e-4, 1e-5],
    'dropout'        : [0.3, 0.5],
    'focal_gamma'    : [1.0, 2.0],
    'focal_alpha'    : [0.50, 0.75],
    'epochs'         : [400, 700, 1000],
}

CV_FOLDS   = 5
ES_PAT_CV  = 20
EVAL_EVERY = 5

keys   = list(PARAM_GRID.keys())
combos = [dict(zip(keys, vals))
          for vals in itertools.product(*PARAM_GRID.values())]

print(f'{"="*60}')
print(f'  5-Fold CV Hyperparameter Search')
print(f'  Combinations : {len(combos)}')
print(f'  Total fits   : {len(combos) * CV_FOLDS}')
print(f'  Device       : {device}')
print(f'{"="*60}\n')

# All data stays on CPU until absolutely needed
y_train_np = y.cpu().numpy().ravel()[:n_train]
train_indices = np.arange(n_train)

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)

# Create folds as plain numpy arrays, keep on CPU
# Only move to GPU INSIDE build_and_train() where we can catch errors
folds = [
    (tr_idx.copy(), va_idx.copy())  # numpy arrays, stay on CPU
    for tr_idx, va_idx in skf.split(train_indices, y_train_np)
]

print(f'  Created {CV_FOLDS} folds (numpy arrays).')
print('  Validating labels on CPU...')

for fi, (tr_idx, va_idx) in enumerate(folds):
    y_tr = y.cpu().numpy().ravel()[tr_idx]
    y_va = y.cpu().numpy().ravel()[va_idx]

    # NaN/Inf check
    assert not np.isnan(y_tr).any(), f'Fold {fi} train NaN'
    assert not np.isinf(y_tr).any(), f'Fold {fi} train Inf'
    assert not np.isnan(y_va).any(), f'Fold {fi} val NaN'
    assert not np.isinf(y_va).any(), f'Fold {fi} val Inf'

    # Binary check
    unique = np.unique(y_va)
    assert list(unique) == [0.0, 1.0], f'Fold {fi} val not binary: {unique}'
    assert len(unique) == 2, f'Fold {fi} val single-class'

print('  All fold labels valid.\n')

# Grid search loop
cv_results     = []
t_search_start = time.time()

for combo_idx, hp in enumerate(combos):
    fold_aucs = []

    for fold_tr_idx, fold_va_idx in folds:
        # Convert to tensors INSIDE build_and_train (where we can catch GPU errors)
        tr_idx_tensor = torch.tensor(fold_tr_idx, dtype=torch.long)
        va_idx_tensor = torch.tensor(fold_va_idx, dtype=torch.long)

        auc = build_and_train(
            hp          = hp,
            tr_idx      = tr_idx_tensor,
            val_idx     = va_idx_tensor,
            es_patience = ES_PAT_CV,
            eval_every  = EVAL_EVERY,
            verbose     = False,
        )
        fold_aucs.append(auc)

        # Clean up after each fold
        if IS_CUDA:
            try:
                torch.cuda.empty_cache()
            except:
                pass

    mean_auc = float(np.mean(fold_aucs))
    std_auc  = float(np.std(fold_aucs))

    cv_results.append({
        **hp,
        'mean_val_AUC': mean_auc,
        'std_val_AUC' : std_auc,
    })

    elapsed_min = (time.time() - t_search_start) / 60
    status = '✔' if mean_auc > 0.5 else '✗'
    print(f'  {status} [{combo_idx + 1:>4}/{len(combos)}]  '
          f'AUC={mean_auc:.4f}±{std_auc:.4f}  '
          f'H={hp["hidden_channels"]}  '
          f'lr={hp["lr"]}  wd={hp["weight_decay"]}  '
          f'do={hp["dropout"]}  γ={hp["focal_gamma"]}  '
          f'α={hp["focal_alpha"]}  ep={hp["epochs"]}  '
          f't={elapsed_min:.1f}min')

# Save results
cv_df  = pd.DataFrame(cv_results).sort_values('mean_val_AUC', ascending=False)
cv_csv = os.path.join(OUT_DIR, 'FGC_GNN_ROSMAP_cv_results.csv')
cv_df.to_csv(cv_csv, index=False)

print(f'\nTop 5:')
print(cv_df[['hidden_channels','lr','mean_val_AUC']].head(5).to_string(index=False))
print(f'\nCV results → {cv_csv}')
print('Cell 8 complete.\n')

  5-Fold CV Hyperparameter Search
  Combinations : 288
  Total fits   : 1440
  Device       : cuda

  Created 5 folds (numpy arrays).
  Validating labels on CPU...
  All fold labels valid.

  ✔ [   1/288]  AUC=0.9069±0.0215  H=128  lr=0.001  wd=0.0001  do=0.3  γ=1.0  α=0.5  ep=400  t=0.6min
  ✔ [   2/288]  AUC=0.8729±0.0254  H=128  lr=0.001  wd=0.0001  do=0.3  γ=1.0  α=0.5  ep=700  t=1.2min
  ✔ [   3/288]  AUC=0.8852±0.0141  H=128  lr=0.001  wd=0.0001  do=0.3  γ=1.0  α=0.5  ep=1000  t=1.5min
  ✔ [   4/288]  AUC=0.8994±0.0138  H=128  lr=0.001  wd=0.0001  do=0.3  γ=1.0  α=0.75  ep=400  t=2.2min
  ✔ [   5/288]  AUC=0.9040±0.0241  H=128  lr=0.001  wd=0.0001  do=0.3  γ=1.0  α=0.75  ep=700  t=2.7min
  ✔ [   6/288]  AUC=0.8779±0.0472  H=128  lr=0.001  wd=0.0001  do=0.3  γ=1.0  α=0.75  ep=1000  t=3.1min
  ✔ [   7/288]  AUC=0.8909±0.0421  H=128  lr=0.001  wd=0.0001  do=0.3  γ=2.0  α=0.5  ep=400  t=3.6min
  ✔ [   8/288]  AUC=0.8939±0.0331  H=128  lr=0.001  wd=0.0001  do=0.3  γ=2.0  α=0.5  ep=700

# Extract Best Hyperparameters From CV Results

In [11]:
best_row = cv_df.iloc[0].to_dict()


best_hp = {
    'hidden_channels': int(best_row['hidden_channels']),
    'lr'             : float(best_row['lr']),
    'weight_decay'   : float(best_row['weight_decay']),
    'dropout'        : float(best_row['dropout']),
    'focal_gamma'    : float(best_row['focal_gamma']),
    'focal_alpha'    : float(best_row['focal_alpha']),
    'epochs'         : int(best_row['epochs']),
    'mean_val_AUC'   : float(best_row['mean_val_AUC']),
    'std_val_AUC'    : float(best_row['std_val_AUC']),
}

print(f'{"="*60}')
print(f'  Best HPs  (mean CV val AUC = {best_hp["mean_val_AUC"]:.4f} '
      f'± {best_hp["std_val_AUC"]:.4f})')
print(f'{"="*60}')
for k, v in best_hp.items():
    if k not in ('mean_val_AUC', 'std_val_AUC'):
        print(f'  {k:<20}: {v}')
print(f'{"="*60}')
print('Cell 9 complete.\n')

  Best HPs  (mean CV val AUC = 0.9269 ± 0.0168)
  hidden_channels     : 128
  lr                  : 0.001
  weight_decay        : 0.0001
  dropout             : 0.5
  focal_gamma         : 1.0
  focal_alpha         : 0.75
  epochs              : 1000
Cell 9 complete.



# Final Training Setup (Test Nodes Remain Locked)

In [12]:
DATASET      = 'ROSMAP'
ES_PATIENCE  = 40
ES_MIN_DELTA = 1e-4
EVAL_EVERY   = 5
EPOCHS       = best_hp['epochs']


local_idx = np.arange(n_train)

local_tr, local_val = train_test_split(
    local_idx,
    test_size  = 0.20,
    random_state = 42,
    stratify   = y_train_np,
)

global_tr_es  = torch.tensor(local_tr,  dtype=torch.long, device=device)
global_val_es = torch.tensor(local_val, dtype=torch.long, device=device)

y_tr_gpu  = y_gpu[global_tr_es]
y_val_gpu = y_gpu[global_val_es]

print(f'  ES-train : {len(global_tr_es):>4} patients  '
      f'(≈{100 * len(global_tr_es) / n_total:.0f}% of total)')
print(f'  ES-val   : {len(global_val_es):>4} patients  '
      f'(ES monitor only, ≈{100 * len(global_val_es) / n_total:.0f}%)')
print(f'  Test     : {len(te_global):>4} patients  (locked _te files)')

# Build final model
final_model = FGC_GNN(
    in_dims         = in_dims,
    hidden_channels = best_hp['hidden_channels'],
    out_channels    = 1,
    dropout         = best_hp['dropout'],
).to(device)


if IS_CUDA and hasattr(torch, 'compile'):
    try:
        final_model = torch.compile(
            final_model, mode='reduce-overhead', fullgraph=False)
        print('\n  torch.compile ')
    except Exception:
        print('\n  torch.compile skipped → eager mode')

# Optimiser, scheduler, loss, scaler, ES
optimizer  = optim.AdamW(
    final_model.parameters(),
    lr           = best_hp['lr'],
    weight_decay = best_hp['weight_decay'],
)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion  = FocalLoss(
    gamma = best_hp['focal_gamma'],
    alpha = best_hp['focal_alpha'],
)
amp_scaler = torch.amp.GradScaler('cuda', enabled=IS_CUDA)
es         = EarlyStopping(
    patience  = ES_PATIENCE,
    min_delta = ES_MIN_DELTA,
    mode      = 'max',
)

print(f'\n  Epochs (max)  : {EPOCHS}')
print(f'  ES patience   : {ES_PATIENCE} eval checks × {EVAL_EVERY} ep')
print('Cell 10 complete.\n')

  ES-train :  195 patients  (≈56% of total)
  ES-val   :   49 patients  (ES monitor only, ≈14%)
  Test     :  105 patients  (locked _te files)

  torch.compile 

  Epochs (max)  : 1000
  ES patience   : 40 eval checks × 5 ep
Cell 10 complete.



# Final Training Loop

In [13]:
epoch_log  = []
stopped_at = EPOCHS

print(f'{"="*60}')
print(f'  Final Training — {DATASET}')
print(f'{"="*60}\n')

pbar = tqdm(range(EPOCHS),
            desc         = f'FGC-GNN ({DATASET})',
            unit         = 'ep',
            dynamic_ncols = True)

for epoch in pbar:

    # TRAIN STEP
    final_model.train()
    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast('cuda', enabled=IS_CUDA):
        out_all = final_model(X_list_gpu, edge_idx_gpu, edge_wt_gpu)

    loss = criterion(out_all[global_tr_es], y_tr_gpu)

    amp_scaler.scale(loss).backward()
    torch.nn.utils.clip_grad_norm_(final_model.parameters(), 2.0)
    amp_scaler.step(optimizer)
    amp_scaler.update()
    scheduler.step()

    loss_val = loss.item()


    if (epoch + 1) % EVAL_EVERY == 0 or epoch == EPOCHS - 1:

        with torch.inference_mode():
            out_e    = final_model(X_list_gpu, edge_idx_gpu, edge_wt_gpu)
            tr_probs = out_e[global_tr_es].view(-1)
            ev_probs = out_e[global_val_es].view(-1)
            te_probs = out_e[te_global].view(-1)

        tr_auc, tr_f1, tr_acc, tr_pr, tr_rc = gpu_metrics(tr_probs, y_tr_gpu)
        ev_auc, ev_f1, ev_acc, ev_pr, ev_rc = gpu_metrics(ev_probs, y_val_gpu)
        te_auc, te_f1, te_acc, te_pr, te_rc = gpu_metrics(te_probs, y_te_gpu)

        epoch_log.append({
            'Epoch'    : epoch + 1,
            'Loss'     : round(loss_val, 6),
            'Train_AUC': round(tr_auc, 4), 'Train_F1' : round(tr_f1, 4),
            'Train_Acc': round(tr_acc, 4),
            'Val_AUC'  : round(ev_auc, 4), 'Val_F1'   : round(ev_f1, 4),
            'Val_Acc'  : round(ev_acc, 4),
            'Test_AUC' : round(te_auc, 4), 'Test_F1'  : round(te_f1, 4),
            'Test_Acc' : round(te_acc, 4),
        })

        pbar.set_postfix({
            'loss'   : f'{loss_val:.4f}',
            'val_AUC': f'{ev_auc:.4f}',
            'te_AUC' : f'{te_auc:.4f}',
            'te_F1'  : f'{te_f1:.4f}',
        })

        # Early stopping check
        if es.step(ev_auc, final_model, epoch):
            stopped_at = epoch + 1
            print(f'\n Early stopping at epoch {stopped_at}  '
                  f'(best val AUC = {es.best_score:.4f} '
                  f'@ epoch {es.best_epoch + 1})')
            break


es.restore_best(final_model)
print(f'  Best checkpoint restored from epoch {es.best_epoch + 1}')
print('Cell 11 complete.\n')

  Final Training — ROSMAP



FGC-GNN (ROSMAP):  26%|██▋       | 264/1000 [01:26<04:01,  3.04ep/s, loss=0.0000, val_AUC=0.7458, te_AUC=0.8108, te_F1=0.7184]


 Early stopping at epoch 265  (best val AUC = 0.8729 @ epoch 65)
  Best checkpoint restored from epoch 65
Cell 11 complete.



# Final Test Evaluation & Save Results

In [14]:
# Test evaluation — one time only
final_model.eval()
with torch.inference_mode():
    out_final = final_model(X_list_gpu, edge_idx_gpu, edge_wt_gpu)
    te_probs  = out_final[te_global].view(-1)

final_auc, final_f1, final_acc, final_prec, final_rec = gpu_metrics(
    te_probs, y_te_gpu)

# Save model checkpoint
ckpt_path = os.path.join(OUT_DIR, f'FGC_{DATASET}_GNN_best.pt')
torch.save(final_model.state_dict(), ckpt_path)

# Per-epoch metrics CSV
epoch_csv = os.path.join(OUT_DIR, f'FGC_GNN_{DATASET}_epoch_metrics.csv')
pd.DataFrame(epoch_log).to_csv(epoch_csv, index=False)

# Summary CSV
summary_row = {
    'Dataset'              : DATASET,
    'AUC'                  : final_auc,
    'F1'                   : final_f1,
    'Accuracy'             : final_acc,
    'Precision'            : final_prec,
    'Recall'               : final_rec,
    'Stopped_Epoch'        : stopped_at,
    'Best_Val_AUC'         : es.best_score,
    'Best_Checkpoint_Epoch': es.best_epoch + 1,
    'ES_train_nodes'       : len(global_tr_es),
    'ES_val_nodes'         : len(global_val_es),
    'Test_nodes'           : len(te_global),
    'CV_mean_AUC'          : best_hp['mean_val_AUC'],
    'CV_std_AUC'           : best_hp['std_val_AUC'],
    **{f'HP_{k}': v
       for k, v in best_hp.items()
       if k not in ('mean_val_AUC', 'std_val_AUC')},
}
summary_csv = os.path.join(OUT_DIR, f'FGC_GNN_{DATASET}_summary.csv')
pd.DataFrame([summary_row]).to_csv(summary_csv, index=False)

# Console summary
print(f'\n{"="*55}')
print(f'  Final Test-Set Results  —  {DATASET}')
print(f'{"="*55}')
print(f'  AUC       : {final_auc:.4f}')
print(f'  F1 Score  : {final_f1:.4f}')
print(f'  Accuracy  : {final_acc:.4f}')
print(f'  Precision : {final_prec:.4f}')
print(f'  Recall    : {final_rec:.4f}')
print(f'{"─"*55}')
print(f'  Stopped at epoch          : {stopped_at}')
print(f'  Best checkpoint epoch     : {es.best_epoch + 1}')
print(f'  Best ES-val AUC           : {es.best_score:.4f}')
print(f'{"─"*55}')
print(f'  Best HPs from CV:')
for k, v in best_hp.items():
    if k not in ('mean_val_AUC', 'std_val_AUC'):
        print(f'    {k:<22}: {v}')
print(f'{"="*55}')
print(f'\nCV results   → {os.path.join(OUT_DIR, "FGC_GNN_ROSMAP_cv_results.csv")}')
print(f'Epoch log    → {epoch_csv}')
print(f'Summary      → {summary_csv}')
print(f'Model ckpt   → {ckpt_path}')
print('Cell 12 complete.\n')


  Final Test-Set Results  —  ROSMAP
  AUC       : 0.8885
  F1 Score  : 0.8073
  Accuracy  : 0.8000
  Precision : 0.8000
  Recall    : 0.8148
───────────────────────────────────────────────────────
  Stopped at epoch          : 265
  Best checkpoint epoch     : 65
  Best ES-val AUC           : 0.8729
───────────────────────────────────────────────────────
  Best HPs from CV:
    hidden_channels       : 128
    lr                    : 0.001
    weight_decay          : 0.0001
    dropout               : 0.5
    focal_gamma           : 1.0
    focal_alpha           : 0.75
    epochs                : 1000

CV results   → ROSMAP_output/FGC_GNN_ROSMAP_cv_results.csv
Epoch log    → ROSMAP_output/FGC_GNN_ROSMAP_epoch_metrics.csv
Summary      → ROSMAP_output/FGC_GNN_ROSMAP_summary.csv
Model ckpt   → ROSMAP_output/FGC_ROSMAP_GNN_best.pt
Cell 12 complete.



#  10-Run Repeated Test Evaluation on 20% Test Subset

In [20]:
import os
import copy
import random
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.optim as optim
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# CONFIG

DATASET      = 'ROSMAP'

SEEDS        = [42,84,64,112,44,128,256,68,52,100]

ES_PATIENCE  = 40
ES_MIN_DELTA = 1e-4
EVAL_EVERY   = 5
EPOCHS       = best_hp['epochs']

RESULTS_ALL  = []
ALL_EPOCH_LOGS = []


# SEED FUNCTION

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Loop Over 10 Seeds
for seed in SEEDS:

    print(f'\n{"="*70}')
    print(f'                 RUNNING SEED = {seed}')
    print(f'{"="*70}')

    # FIX RANDOMNESS

    set_seed(seed)


    # EARLY-STOPPING SPLIT

    local_idx = np.arange(n_train)

    local_tr, local_val = train_test_split(
        local_idx,
        test_size   = 0.20,
        random_state= seed,
        stratify    = y_train_np,
    )

    global_tr_es  = torch.tensor(local_tr,  dtype=torch.long, device=device)
    global_val_es = torch.tensor(local_val, dtype=torch.long, device=device)

    y_tr_gpu  = y_gpu[global_tr_es]
    y_val_gpu = y_gpu[global_val_es]

    print(f'  ES-train : {len(global_tr_es)}')
    print(f'  ES-val   : {len(global_val_es)}')
    print(f'  Test     : {len(te_global)}')


    # BUILD MODEL

    final_model = FGC_GNN(
        in_dims         = in_dims,
        hidden_channels = best_hp['hidden_channels'],
        out_channels    = 1,
        dropout         = best_hp['dropout'],
    ).to(device)

    # torch.compile (optional)
    if IS_CUDA and hasattr(torch, 'compile'):
        try:
            final_model = torch.compile(
                final_model,
                mode='reduce-overhead',
                fullgraph=False
            )
            print('  torch.compile enabled')
        except Exception:
            print('  torch.compile skipped')


    # OPTIMIZER / LOSS / SCHEDULER

    optimizer = optim.AdamW(
        final_model.parameters(),
        lr           = best_hp['lr'],
        weight_decay = best_hp['weight_decay'],
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    criterion = FocalLoss(
        gamma = best_hp['focal_gamma'],
        alpha = best_hp['focal_alpha'],
    )

    amp_scaler = torch.amp.GradScaler(
        'cuda',
        enabled=IS_CUDA
    )

    es = EarlyStopping(
        patience  = ES_PATIENCE,
        min_delta = ES_MIN_DELTA,
        mode      = 'max',
    )


    # TRAINING LOOP

    epoch_log  = []
    stopped_at = EPOCHS

    pbar = tqdm(
        range(EPOCHS),
        desc=f'Seed {seed}',
        unit='ep',
        dynamic_ncols=True
    )

    for epoch in pbar:


        # TRAIN

        final_model.train()

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=IS_CUDA):

            out_all = final_model(
                X_list_gpu,
                edge_idx_gpu,
                edge_wt_gpu
            )

        loss = criterion(
            out_all[global_tr_es],
            y_tr_gpu
        )

        amp_scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            final_model.parameters(),
            2.0
        )

        amp_scaler.step(optimizer)
        amp_scaler.update()

        scheduler.step()

        loss_val = loss.item()


        # EVALUATION

        if (epoch + 1) % EVAL_EVERY == 0 or epoch == EPOCHS - 1:

            final_model.eval()

            with torch.inference_mode():

                out_e = final_model(
                    X_list_gpu,
                    edge_idx_gpu,
                    edge_wt_gpu
                )

                tr_probs = out_e[global_tr_es].view(-1)
                ev_probs = out_e[global_val_es].view(-1)
                te_probs = out_e[te_global].view(-1)

            tr_auc, tr_f1, tr_acc, tr_pr, tr_rc = gpu_metrics(
                tr_probs, y_tr_gpu)

            ev_auc, ev_f1, ev_acc, ev_pr, ev_rc = gpu_metrics(
                ev_probs, y_val_gpu)

            te_auc, te_f1, te_acc, te_pr, te_rc = gpu_metrics(
                te_probs, y_te_gpu)

            epoch_log.append({
                'Seed'      : seed,
                'Epoch'     : epoch + 1,
                'Loss'      : round(loss_val, 6),

                'Train_AUC' : round(tr_auc, 4),
                'Train_F1'  : round(tr_f1, 4),
                'Train_Acc' : round(tr_acc, 4),

                'Val_AUC'   : round(ev_auc, 4),
                'Val_F1'    : round(ev_f1, 4),
                'Val_Acc'   : round(ev_acc, 4),

                'Test_AUC'  : round(te_auc, 4),
                'Test_F1'   : round(te_f1, 4),
                'Test_Acc'  : round(te_acc, 4),
            })

            pbar.set_postfix({
                'loss'    : f'{loss_val:.4f}',
                'val_auc' : f'{ev_auc:.4f}',
                'test_auc': f'{te_auc:.4f}',
            })


            # EARLY STOPPING

            if es.step(ev_auc, final_model, epoch):

                stopped_at = epoch + 1

                print(
                    f'\n Early stopping @ epoch {stopped_at} '
                    f'(best val AUC = {es.best_score:.4f})'
                )

                break


    # RESTORE BEST MODEL

    es.restore_best(final_model)

    print(f'  Restored best checkpoint from epoch '
          f'{es.best_epoch + 1}')


    # FINAL TEST EVALUATION

    final_model.eval()

    with torch.inference_mode():

        out_final = final_model(
            X_list_gpu,
            edge_idx_gpu,
            edge_wt_gpu
        )

        te_probs = out_final[te_global].view(-1)

    final_auc, final_f1, final_acc, final_prec, final_rec = gpu_metrics(
        te_probs,
        y_te_gpu
    )


    # SAVE CHECKPOINT

    ckpt_path = os.path.join(
        OUT_DIR,
        f'FGC_{DATASET}_seed_{seed}.pt'
    )

    torch.save(final_model.state_dict(), ckpt_path)


    # STORE RESULTS

    result_row = {

        'Seed'               : seed,

        'AUC'                : final_auc,
        'F1'                 : final_f1,
        'Accuracy'           : final_acc,
        'Precision'          : final_prec,
        'Recall'             : final_rec,

        'Stopped_Epoch'      : stopped_at,
        'Best_Val_AUC'       : es.best_score,
        'Best_Epoch'         : es.best_epoch + 1,
    }

    RESULTS_ALL.append(result_row)

    ALL_EPOCH_LOGS.extend(epoch_log)


    # PRINT RESULTS

    print(f'\n{"-"*60}')
    print(f' SEED {seed} RESULTS')
    print(f'{"-"*60}')
    print(f' AUC       : {final_auc:.4f}')
    print(f' F1        : {final_f1:.4f}')
    print(f' Accuracy  : {final_acc:.4f}')
    print(f' Precision : {final_prec:.4f}')
    print(f' Recall    : {final_rec:.4f}')
    print(f'{"-"*60}')


# Save Final Results

results_df = pd.DataFrame(RESULTS_ALL)


# SAVE SEED RESULTS

results_csv = os.path.join(
    OUT_DIR,
    f'FGC_GNN_{DATASET}_10Seed_Results.csv'
)

results_df.to_csv(results_csv, index=False)


# SAVE EPOCH LOGS

epoch_logs_csv = os.path.join(
    OUT_DIR,
    f'FGC_GNN_{DATASET}_10Seed_EpochLogs.csv'
)

pd.DataFrame(ALL_EPOCH_LOGS).to_csv(
    epoch_logs_csv,
    index=False
)


# SUMMARY STATISTICS

summary_stats = {

    'Metric' : [
        'AUC',
        'F1',
        'Accuracy',
        'Precision',
        'Recall'
    ],

    'Mean' : [
        results_df['AUC'].mean(),
        results_df['F1'].mean(),
        results_df['Accuracy'].mean(),
        results_df['Precision'].mean(),
        results_df['Recall'].mean(),
    ],

    'Std' : [
        results_df['AUC'].std(),
        results_df['F1'].std(),
        results_df['Accuracy'].std(),
        results_df['Precision'].std(),
        results_df['Recall'].std(),
    ]
}

summary_df = pd.DataFrame(summary_stats)

summary_csv = os.path.join(
    OUT_DIR,
    f'FGC_GNN_{DATASET}_10Seed_Summary.csv'
)

summary_df.to_csv(summary_csv, index=False)


# Final Outputs

print(f'\n{"="*70}')
print(f'            FINAL 10-SEED RESULTS — {DATASET}')
print(f'{"="*70}')

print(results_df)

print(f'\n{"-"*70}')
print(' MEAN ± STD')
print(f'{"-"*70}')

for metric in ['AUC', 'F1', 'Accuracy', 'Precision', 'Recall']:

    mean_val = results_df[metric].mean()
    std_val  = results_df[metric].std()

    print(f' {metric:<10}: {mean_val:.4f} ± {std_val:.4f}')

print(f'\n{"="*70}')
print(f' Saved files:')
print(f'{"="*70}')
print(f' Seed Results : {results_csv}')
print(f' Epoch Logs   : {epoch_logs_csv}')
print(f' Summary CSV  : {summary_csv}')
print(f'{"="*70}')


                 RUNNING SEED = 42
  ES-train : 195
  ES-val   : 49
  Test     : 105
  torch.compile enabled


Seed 42:  25%|██▍       | 249/1000 [00:03<00:09, 77.01ep/s, loss=0.0000, val_auc=0.8495, test_auc=0.8588]



 Early stopping @ epoch 250 (best val AUC = 0.8629)
  Restored best checkpoint from epoch 50

------------------------------------------------------------
 SEED 42 RESULTS
------------------------------------------------------------
 AUC       : 0.8809
 F1        : 0.8000
 Accuracy  : 0.7810
 Precision : 0.7541
 Recall    : 0.8519
------------------------------------------------------------

                 RUNNING SEED = 84
  ES-train : 195
  ES-val   : 49
  Test     : 105
  torch.compile enabled


Seed 84:  23%|██▎       | 229/1000 [00:03<00:10, 74.08ep/s, loss=0.0000, val_auc=0.8528, test_auc=0.8867]



 Early stopping @ epoch 230 (best val AUC = 0.8913)
  Restored best checkpoint from epoch 30

------------------------------------------------------------
 SEED 84 RESULTS
------------------------------------------------------------
 AUC       : 0.8715
 F1        : 0.7706
 Accuracy  : 0.7619
 Precision : 0.7636
 Recall    : 0.7778
------------------------------------------------------------

                 RUNNING SEED = 64
  ES-train : 195
  ES-val   : 49
  Test     : 105
  torch.compile enabled


...


 Early stopping @ epoch 205 (best val AUC = 0.8679)
  Restored best checkpoint from epoch 5

------------------------------------------------------------
 SEED 68 RESULTS
------------------------------------------------------------
 AUC       : 0.8243
 F1        : 0.7216
 Accuracy  : 0.7429
 Precision : 0.8140
 Recall    : 0.6481
------------------------------------------------------------

                 RUNNING SEED = 52
  ES-train : 195
  ES-val   : 49
  Test     : 105
  torch.compile enabled


Seed 52:  26%|██▌       | 259/1000 [00:03<00:09, 78.81ep/s, loss=0.0000, val_auc=0.8829, test_auc=0.8301]



 Early stopping @ epoch 260 (best val AUC = 0.9114)
  Restored best checkpoint from epoch 60

------------------------------------------------------------
 SEED 52 RESULTS
------------------------------------------------------------
 AUC       : 0.8460
 F1        : 0.7290
 Accuracy  : 0.7238
 Precision : 0.7358
 Recall    : 0.7222
------------------------------------------------------------

                 RUNNING SEED = 100
  ES-train : 195
  ES-val   : 49
  Test     : 105
  torch.compile enabled


Seed 100:  39%|███▉      | 389/1000 [00:04<00:07, 84.81ep/s, loss=0.0000, val_auc=0.8328, test_auc=0.8588]



 Early stopping @ epoch 390 (best val AUC = 0.8395)
  Restored best checkpoint from epoch 190

------------------------------------------------------------
 SEED 100 RESULTS
------------------------------------------------------------
 AUC       : 0.8660
 F1        : 0.7928
 Accuracy  : 0.7810
 Precision : 0.7719
 Recall    : 0.8148
------------------------------------------------------------

            FINAL 10-SEED RESULTS — ROSMAP
   Seed       AUC        F1  Accuracy  Precision    Recall  Stopped_Epoch  \
0    42  0.880901  0.800000  0.780952   0.754098  0.851852            250   
1    84  0.871460  0.770642  0.761905   0.763636  0.777778            230   
2    64  0.857298  0.754717  0.752381   0.769231  0.740741            405   
3   112  0.858751  0.777778  0.771429   0.777778  0.777778            495   
4    44  0.851852  0.761905  0.761905   0.784314  0.740741            235   
5   128  0.863471  0.774775  0.761905   0.754386  0.796296            225   
6   256  0.843863  0